# Apeireth-Decis Training
**Qwen3-0.6B → 8-expert MoE decision model**

⚠️ Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install -q "transformers>=4.55,<5" datasets bitsandbytes accelerate safetensors pyarrow pillow

## 1. Download code & data

In [ ]:
import os, subprocess, sys, time, json, urllib.request
os.environ["HF_HOME"] = "/content/hf"

import torch
assert torch.cuda.is_available(), "需要 GPU！"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 2**30:.1f} GB")

RAW = "https://raw.githubusercontent.com/xiaocongyu66/mistral.rs/decide/distill"
os.makedirs("/content/distill", exist_ok=True)
for f in ["colab_train.py", "upcycle_to_moe.py", "stream_sources.py"]:
    urllib.request.urlretrieve(f"{RAW}/{f}", f"/content/distill/{f}")
    print("fetched", f)

from huggingface_hub import hf_hub_download
import shutil
p = hf_hub_download("congyu778/duan-distill", "unified.jsonl", repo_type="dataset")
os.makedirs("/content/distill/nanojev_format", exist_ok=True)
shutil.copy(p, "/content/distill/nanojev_format/unified.jsonl")
print("data ready:", os.path.getsize("/content/distill/nanojev_format/unified.jsonl") // 1e6, "MB")

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/TianyuCodings/NanoJev/main/scripts/train_toy_decisions.py",
    "/content/distill/train_toy_decisions.py")
print("all ready")

## 2. Train (600 steps, ~40 min)

In [ ]:
sys.path.insert(0, "/content/distill")
sys.argv = ["colab_train.py",
    "--nanojev-scripts", "/content/distill",
    "--input", "/content/distill/nanojev_format/unified.jsonl",
    "--output-dir", "/content/apeireth-decis-2.6b-128k",
    "--model", "Qwen/Qwen3-0.6B",
    "--steps", "600", "--head-steps", "24", "--eval-every", "50",
    "--batch-questions", "4",
    "--temperature", "5.0", "--temperature-final", "1.5",
    "--entropy-weighting", "--dtype", "fp16", "--adam8bit"]

exec(open("/content/distill/colab_train.py").read())

## 3. Save to Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('/content/apeireth-decis-2.6b-128k/best.safetensors',
            '/content/drive/MyDrive/Apeireth-Decis-2.6B-best.safetensors')
shutil.copy('/content/apeireth-decis-2.6b-128k/summary.json',
            '/content/drive/MyDrive/Apeireth-Decis-summary.json')
print("saved to Google Drive")